# max-back-tied-half — worked example 2: Compare Half-Mass ReLU Backward vs PyTorch's Strict Convention

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `max-back-tied-half`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

ReLU is `maximum(x, 0)`, so `relu_back(grad_out, x)` is a special case of `maximum_back0` with `y = zeros_like(x)`. The half-mass convention assigns `0.5 * grad_out` at the kink `x == 0`, while PyTorch's built-in uses strict inequality (`x > 0`), giving zero gradient at the kink. Both are valid subgradients; the difference is visible only when input values hit zero exactly.

## Worked solution

**Step 1 — define both conventions.**
The half-mass relu_back uses `(x > 0) + 0.5*(x == 0)` as the mask. PyTorch's convention uses `(x > 0)` only.

**Step 2 — construct an input with a zero exactly at position 2.**
This makes the kink case explicit and testable.

**Step 3 — compare the outputs at each position.**
At `x > 0`: both agree — gradient passes through. At `x < 0`: both agree — gradient is zero. At `x == 0`: half-mass gives `0.5 * grad_out`, PyTorch gives `0`.

**Step 4 — verify against torch.nn.functional.relu.**
We run a forward pass with `requires_grad=True` and check which convention torch uses at zero.

In [ ]:
import torch as t

def relu_back_half_mass(grad_out, x):
    """Half-mass convention: 0.5 at kink x==0."""
    mask = (x > 0).to(grad_out.dtype) + 0.5 * (x == 0).to(grad_out.dtype)
    return grad_out * mask

def relu_back_strict(grad_out, x):
    """Strict convention (PyTorch default): 0 at kink."""
    return grad_out * (x > 0).to(grad_out.dtype)

# Input with a zero exactly at position 1
x = t.tensor([-1.0, 0.0, 2.0, -0.5, 3.0])
grad_out = t.ones(5)

g_half = relu_back_half_mass(grad_out, x)
g_strict = relu_back_strict(grad_out, x)

print("x:           ", x.tolist())
print("half-mass:   ", g_half.tolist())
print("strict:      ", g_strict.tolist())
print("\nDifference only at x==0:")
print("  half-mass[1] =", g_half[1].item(), " (0.5)")
print("  strict[1]    =", g_strict[1].item(), " (0.0)")

# Verify pytorch uses strict convention
x_ag = x.clone().requires_grad_(True)
out = t.relu(x_ag)
loss = (out * grad_out).sum()
loss.backward()
pytorch_grad = x_ag.grad
print(f"\nPyTorch ReLU grad at x=0: {pytorch_grad[1].item()}  (strict convention)")
print(f"Agree at non-zero positions: {t.allclose(g_half[[0,2,3,4]], pytorch_grad[[0,2,3,4]])}")